In [1]:
import os
import json
import time
import random
import re
from datetime import date
from pathlib import Path

from google import genai
from google.genai import types, errors

In [2]:
API_KEY = os.environ.get("GEMINI_API_KEY")
if not API_KEY:
    raise RuntimeError(
        "Nema promenljive okruženja GEMINI_API_KEY. Postavi je sa "
        'setx GEMINI_API_KEY "tvoj_kljuc" i ponovo pokreni VS Code.'
    )

client = genai.Client(api_key=API_KEY)
print("Klijent napravljen.")

Klijent napravljen.


In [3]:
MODEL = "gemini-3.5-flash-lite"
TEMPERATURA = 0.9
SEED = 42
PAUZA_SEK = 20
N_VRSTA = None

MAX_POKUSAJA_API = 10
MAX_POKUSAJA_IME = 3

BACKEND_DIR = Path("..").resolve()
KB_DIR = BACKEND_DIR / "knowledge_base"
KB_JSON = KB_DIR / "knowledge_base.json"
IZLAZ_JSON = KB_DIR / "tekst_upiti.json"
print("Ulaz:", KB_JSON)
print("Izlaz:", IZLAZ_JSON)

In [4]:
for m in client.models.list():
    if "generateContent" in (m.supported_actions or []):
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyria-3.5
models/gemini-3.1-flash-tts-preview
models/

In [5]:
PROMPT_VERZIJA = "v2"

SYSTEM_PROMPT = """You help build a test set for a plant identification system.

You will receive a botanical description of one plant species. Rewrite it as queries that an ordinary person (not a botanist) would type into a search box after seeing this plant in nature.

Rules:
1. Use everyday words instead of botanical terms: "hairy" not "pubescent", "egg-shaped" not
   "ovate", "seed pod" not "capsule", "flower head" not "capitulum". If there is no everyday
   word, describe what the part looks like.
2. Never mention the scientific name, the genus, the family or any common name of the plant,
   and do not hint at them.
3. Use only features stated in the description. Do not add anything from your own knowledge.
4. Prefer features a person can see, touch or smell: height, flower colour and shape, leaves,
   stem, fruit, smell. Skip microscopic details, chromosome numbers and geographic range.
5. Do not follow the order of the description. A person first mentions what caught their eye,
   usually the flower.
6. Write the way people really type: plain sentences, no lists, sometimes approximate
   ("about knee high", "smallish yellow flowers").
7. Describe only the plant. Every query starts directly with the plant or its feature
   ("Tall plant with...", "Shrub with...", "Pink round flower heads on..."). Never use framing
   phrases such as "I found", "I saw", "I spotted", "I noticed", "I am trying to identify",
   "What is this", "Can you tell me".
8. Always write the queries in English.
9. Return exactly:
   - "short": 2 short queries, one sentence each, 2-3 features each;
   - "detailed": 2 detailed queries, 3-5 sentences each, 4-6 features each.
   Do not try to include every feature from the description. Pick the features an ordinary
   person would realistically notice and mention.
   The two queries of the same kind should focus on different features where the description
   allows it.

Answer only with JSON."""

SEMA_ODGOVORA = types.Schema(
    type=types.Type.OBJECT,
    properties={
        "short": types.Schema(
            type=types.Type.ARRAY, items=types.Schema(type=types.Type.STRING),
            min_items=2, max_items=2,
        ),
        "detailed": types.Schema(
            type=types.Type.ARRAY, items=types.Schema(type=types.Type.STRING),
            min_items=2, max_items=2,
        ),
    },
    required=["short", "detailed"],
)

PODESAVANJA_POZIVA = types.GenerateContentConfig(
    system_instruction=SYSTEM_PROMPT,
    temperature=TEMPERATURA,
    response_mime_type="application/json",
    response_schema=SEMA_ODGOVORA,
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)

In [6]:
if not KB_JSON.exists():
    raise FileNotFoundError(f"Ne postoji {KB_JSON}.")
kb = json.loads(KB_JSON.read_text(encoding="utf-8"))
print("Učitan knowledge base:", len(kb), "vrsta.")

bez_teksta = sorted(sid for sid, z in kb.items() if not (z.get("tekst") or "").strip())
print("Vrste bez teksta (preskaču se):", bez_teksta if bez_teksta else "nema")

Učitan knowledge base: 10 vrsta.
Vrste bez teksta (preskaču se): nema


In [7]:
OZNAKA_IZVORA = re.compile(r"\[(?:\d+|[a-z][a-z ]*)\]")
WIKI_NASLOV = re.compile(r"^=+\s*.+?\s*=+$")


def je_naslov(red):
    red = red.strip()
    return bool(red) and len(red.split()) <= 4 and ":" not in red and not re.search(r"[.!?;,)]$", red)


def ocisti_tekst(tekst):
    tekst = OZNAKA_IZVORA.sub("", tekst)
    redovi = []
    for red in tekst.split("\n"):
        if WIKI_NASLOV.match(red.strip()) or je_naslov(red):
            continue
        redovi.append(red)
    tekst = "\n".join(redovi)
    tekst = re.sub(r"(?<=[a-z])\.(?=[A-Z])", ". ", tekst)
    tekst = re.sub(r"\n{3,}", "\n\n", tekst)
    return tekst.strip()


promenjene = []
for sid, zapis in kb.items():
    tekst = zapis.get("tekst") or ""
    ociscen = ocisti_tekst(tekst)
    if ociscen != tekst.strip():
        promenjene.append(sid)
    zapis["tekst"] = ociscen

print("Očišćeno:", len(promenjene), "od", len(kb), "tekstova")
for sid in promenjene:
    print(" ", sid, kb[sid]["scientific_name"])

Očišćeno: 4 od 10 tekstova
  0544 Hypericum olympicum
  0712 Dryopteris intermedia
  0730 Maianthemum stellatum
  0861 Aralia spinosa


In [8]:
PRIVREMENE_GRESKE = {429, 500, 503}


def posalji_upit(tekst):
    poruka = f"Plant description:\n{tekst}"
    for pokusaj in range(1, MAX_POKUSAJA_API + 1):
        try:
            odgovor = client.models.generate_content(
                model=MODEL, contents=poruka, config=PODESAVANJA_POZIVA
            )
            if not odgovor.text:
                raise ValueError("prazan odgovor (model je možda blokirao sadržaj)")
            podaci = json.loads(odgovor.text)
            if len(podaci.get("short", [])) != 2 or len(podaci.get("detailed", [])) != 2:
                raise ValueError(f"očekivano 2 + 2 upita, dobijeno: {podaci}")
            return podaci
        except errors.APIError as e:
            if e.code not in PRIVREMENE_GRESKE or pokusaj == MAX_POKUSAJA_API:
                raise
            cekaj = min(PAUZA_SEK * 2 ** (pokusaj - 1), 60)
            print(f"   API greška {e.code}, čekam {cekaj} s (pokušaj {pokusaj})")
            time.sleep(cekaj)
        except (json.JSONDecodeError, ValueError) as e:
            if pokusaj == MAX_POKUSAJA_API:
                raise
            print(f"   loš odgovor ({e}), pokušavam ponovo")


def sadrzi_ime(upit, scientific_name):
    delovi = scientific_name.lower().split()[:2]
    upit = upit.lower()
    return any(re.search(rf"\b{re.escape(d)}\b", upit) for d in delovi)


def generisi_za_vrstu(zapis):
    for pokusaj in range(1, MAX_POKUSAJA_IME + 1):
        podaci = posalji_upit(zapis["tekst"])
        svi_upiti = podaci["short"] + podaci["detailed"]
        if not any(sadrzi_ime(u, zapis["scientific_name"]) for u in svi_upiti):
            return podaci, False
        print(f"   upit sadrži ime biljke, tražim nove (pokušaj {pokusaj})")
    return podaci, True

In [9]:
N_PROBA = 10
DUZINA_KARTICE = 800

kandidati = sorted(sid for sid in kb if sid not in bez_teksta)
proba_ids = random.Random(SEED).sample(kandidati, min(N_PROBA, len(kandidati)))
proba_rezultati = {}

print(f"Model: {MODEL} | prompt: {PROMPT_VERZIJA} | vrsta: {len(proba_ids)}\n")
for i, sid in enumerate(proba_ids, 1):
    zapis = kb[sid]
    try:
        podaci = posalji_upit(zapis["tekst"])
    except Exception as e:
        print(f"[{i}] {sid} {zapis['scientific_name']}: greška — {e}\n")
        continue
    proba_rezultati[sid] = podaci

    tekst = zapis["tekst"]
    print("=" * 100)
    print(f"[{i}/{len(proba_ids)}] {sid} {zapis['scientific_name']}")
    print("KARTICA:", tekst[:DUZINA_KARTICE] + (" [...]" if len(tekst) > DUZINA_KARTICE else ""))
    print()
    for stil in ("short", "detailed"):
        for upit in podaci[stil]:
            oznaka = "  IME!" if sadrzi_ime(upit, zapis["scientific_name"]) else ""
            print(f"  [{stil}]{oznaka} {upit}")
    print()
    if i < len(proba_ids):
        time.sleep(PAUZA_SEK)

print(f"Gotovo: {len(proba_rezultati)}/{len(proba_ids)} vrsta.")

Model: gemini-3.5-flash-lite | prompt: v2 | vrsta: 10

[1/10] 0489 Sagittaria sagittifolia
KARTICA: Sagittaria sagittifolia is a herbaceous perennial plant, growing in water from 10–50 centimetres (4–19+1⁄2 inches) deep. The leaves above water are arrowhead-shaped, the leaf blade 15–25 cm (6–10 in) long and 10–22 cm (4–8+1⁄2 in) broad, on a long petiole holding the leaf up to 45 cm (17+1⁄2 in) above water level. The plant also has narrow linear submerged leaves, up to 80 cm (31+1⁄2 in) long and 2 cm broad. Panicled flowers are 2–2.5 cm broad, with three small sepals and three white petals, and numerous purple stamens.

  [short] Water plant with arrowhead shaped leaves and white flowers.
  [short] Plant growing in water with white flowers and purple centers.
  [detailed] Aquatic plant with large arrowhead shaped leaves sticking up high above the water surface. It has branched flower stalks bearing small blooms with three white petals and lots of purple centers. The leaves are quite wid

In [11]:
def podeli_val_test(specie_id, podaci):
    rng = random.Random(f"{SEED}-{specie_id}")
    short, detailed = list(podaci["short"]), list(podaci["detailed"])
    rng.shuffle(short)
    rng.shuffle(detailed)

    val = [{"stil": "short", "upit": short[0]}, {"stil": "detailed", "upit": detailed[0]}]
    test = [{"stil": "short", "upit": short[1]}, {"stil": "detailed", "upit": detailed[1]}]
    return val, test


def ucitaj_izlaz():
    meta = {
        "model": MODEL,
        "prompt_verzija": PROMPT_VERZIJA,
        "temperatura": TEMPERATURA,
        "seed": SEED,
        "datum": date.today().isoformat(),
    }
    if not IZLAZ_JSON.exists():
        return {"_meta": meta}
    izlaz = json.loads(IZLAZ_JSON.read_text(encoding="utf-8"))
    stara = izlaz["_meta"]
    if stara["model"] != MODEL or stara["prompt_verzija"] != PROMPT_VERZIJA:
        raise RuntimeError(
            f"{IZLAZ_JSON.name} je napravljen sa modelom {stara['model']} i promptom "
            f"{stara['prompt_verzija']}, a sada je {MODEL} / {PROMPT_VERZIJA}. "
            "Preimenuj ili obriši stari fajl pre novog pokretanja."
        )
    return izlaz


def snimi(izlaz):
    privremeni = IZLAZ_JSON.with_suffix(".tmp")
    privremeni.write_text(json.dumps(izlaz, ensure_ascii=False, indent=2), encoding="utf-8")
    privremeni.replace(IZLAZ_JSON)

In [ ]:
izlaz = ucitaj_izlaz()
for sid, podaci in proba_rezultati.items():
    if sid in izlaz:
        continue
    zapis = kb[sid]
    svi_upiti = podaci["short"] + podaci["detailed"]
    val, test = podeli_val_test(sid, podaci)
    izlaz[sid] = {
        "scientific_name": zapis["scientific_name"],
        "izvor": zapis.get("izvor"),
        "curenje_imena": any(sadrzi_ime(u, zapis["scientific_name"]) for u in svi_upiti),
        "val": val,
        "test": test,
    }
izlaz = {"_meta": izlaz["_meta"], **dict(sorted((k, v) for k, v in izlaz.items() if k != "_meta"))}
snimi(izlaz)

print("Snimljeno u", IZLAZ_JSON.name, "| vrsta:", len(izlaz) - 1)
for sid, z in izlaz.items():
    if sid != "_meta" and z["curenje_imena"]:
        print("  CURENJE IMENA:", sid, z["scientific_name"])

In [17]:

izlaz = ucitaj_izlaz()

za_obradu = [sid for sid in sorted(kb) if sid not in izlaz and sid not in bez_teksta]
print("Već obrađeno:", len(izlaz) - 1, "| preostalo:", len(za_obradu))
if N_VRSTA is not None:
    za_obradu = za_obradu[:N_VRSTA]
print("U ovom pokretanju:", len(za_obradu), "\n")

for i, sid in enumerate(za_obradu, 1):
    zapis = kb[sid]
    print(f"[{i}/{len(za_obradu)}] {sid} {zapis['scientific_name']}")
    try:
        podaci, curenje = generisi_za_vrstu(zapis)
    except Exception as e:
        print(f"\nZaustavljeno kod {sid}: {e}")
        print("Urađeno je snimljeno; pokreni ćeliju ponovo kasnije.")
        break

    val, test = podeli_val_test(sid, podaci)
    izlaz[sid] = {
        "scientific_name": zapis["scientific_name"],
        "izvor": zapis.get("izvor"),
        "curenje_imena": curenje,
        "val": val,
        "test": test,
    }
    snimi(izlaz)
    time.sleep(PAUZA_SEK)

print("\nUkupno vrsta u fajlu:", len(izlaz) - 1)

Već obrađeno: 66 | preostalo: 258
U ovom pokretanju: 258 

[1/258] 0125 Althaea officinalis
   API greška 503, čekam 20 s (pokušaj 1)
[2/258] 0126 Nymphaea alba
   API greška 503, čekam 20 s (pokušaj 1)
   API greška 503, čekam 40 s (pokušaj 2)
[3/258] 0131 Adonis aestivalis
[4/258] 0132 Adonis annua
[5/258] 0135 Anemone coronaria
[6/258] 0138 Hyoscyamus albus
   API greška 503, čekam 20 s (pokušaj 1)
[7/258] 0139 Hyoscyamus niger
[8/258] 0140 Daphne gnidium
[9/258] 0141 Daphne laureola
[10/258] 0143 Berula erecta
[11/258] 0144 Chaerophyllum temulum
[12/258] 0146 Meum athamanticum
[13/258] 0148 Thapsia villosa
[14/258] 0149 Valeriana calcitrapae
[15/258] 0152 Pancratium maritimum
[16/258] 0153 Anthericum liliago
   API greška 503, čekam 20 s (pokušaj 1)
[17/258] 0154 Butomus umbellatus
[18/258] 0158 Scilla peruviana
[19/258] 0161 Ophrys apifera
[20/258] 0163 Ophrys fusca
[21/258] 0164 Ophrys lutea
   API greška 503, čekam 20 s (pokušaj 1)
[22/258] 0165 Ophrys scolopax
[23/258] 0166 Oph